<a href="https://colab.research.google.com/github/Reham-f1/California-housing-querying/blob/main/dataB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Data cleaning completed (California housing dataset). Moving on to SQL queries on the cleaned dataset

In [2]:
import sqlite3
import pandas as pd

# Create a new SQLite database connection (or open it if it already exists)
conn = sqlite3.connect('my_database.db')

# Create a cursor object to execute SQL commands
cursor = conn.cursor()


In [3]:
# Reminder: Never run queries or analysis on raw data directly.
# The dataset used here has been pre-cleaned.
# The cleaning steps were provided in the earlier code.


# Load the cleaned housing data from a CSV file into a pandas DataFrame
dff = pd.read_csv('/content/drive/MyDrive/CLEANDhousing.csv')

# Display the first few rows of the DataFrame to verify successful loading
dff.head()


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [4]:
# Transfer data from the pandas DataFrame to the SQLite database table named 'housing'.
# If the table already exists, replace it. Do not write DataFrame index as a column.
dff.to_sql('housing', conn, if_exists='replace', index=False)

# Execute a query to fetch the first 5 rows from the 'housing' table
cursor.execute("SELECT * FROM housing LIMIT 5")
rows = cursor.fetchall()

# Print each row fetched from the query result
for row in rows:
    print(row)


(-122.23, 37.88, 41.0, 880.0, 129.0, 322.0, 126.0, 8.3252, 452600.0, 'NEAR BAY')
(-122.22, 37.86, 21.0, 7099.0, 1106.0, 2401.0, 1138.0, 8.3014, 358500.0, 'NEAR BAY')
(-122.24, 37.85, 52.0, 1467.0, 190.0, 496.0, 177.0, 7.2574, 352100.0, 'NEAR BAY')
(-122.25, 37.85, 52.0, 1274.0, 235.0, 558.0, 219.0, 5.6431, 341300.0, 'NEAR BAY')
(-122.25, 37.85, 52.0, 1627.0, 280.0, 565.0, 259.0, 3.8462, 342200.0, 'NEAR BAY')


In [5]:
# Retrieve metadata about the 'housing' table structure (column names, types, etc.)
cursor.execute("PRAGMA table_info(housing);")
columns = cursor.fetchall()

# Print out the table's column information
for col in columns:
    print(col)

# Based on the result, the column names are correctly set in the SQLite table,
# and the setup is working as expected.


(0, 'longitude', 'REAL', 0, None, 0)
(1, 'latitude', 'REAL', 0, None, 0)
(2, 'housing_median_age', 'REAL', 0, None, 0)
(3, 'total_rooms', 'REAL', 0, None, 0)
(4, 'total_bedrooms', 'REAL', 0, None, 0)
(5, 'population', 'REAL', 0, None, 0)
(6, 'households', 'REAL', 0, None, 0)
(7, 'median_income', 'REAL', 0, None, 0)
(8, 'median_house_value', 'REAL', 0, None, 0)
(9, 'ocean_proximity', 'TEXT', 0, None, 0)


In [6]:
# Define a helper function to run SQL queries and display the results
def run_query(query):
    cursor.execute(query)
    columns = [desc[0] for desc in cursor.description]  # Get column names
    rows = cursor.fetchall()                            # Fetch all results

    print("🟩 Columns:")
    print(columns)
    for row in rows:
        print(row)


In [7]:

# Example queries using the function:

# Get the total number of rows (i.e., number of houses) in the table
run_query("SELECT COUNT(*) AS total_rows FROM housing")



🟩 Columns:
['total_rows']
(20433,)


In [8]:
# Get the minimum and maximum house prices
run_query("SELECT MIN(median_house_value) AS min_price, MAX(median_house_value) AS max_price FROM housing")


🟩 Columns:
['min_price', 'max_price']
(14999.0, 500001.0)


In [9]:
# Calculate the average house price for each ocean proximity category
run_query("SELECT ocean_proximity, AVG(median_house_value) AS average_price FROM housing GROUP BY ocean_proximity")


🟩 Columns:
['ocean_proximity', 'average_price']
('<1H OCEAN', 240267.99081248615)
('INLAND', 124896.86314655172)
('ISLAND', 380440.0)
('NEAR BAY', 259279.29207048457)
('NEAR OCEAN', 249042.35502283106)


In [10]:
# Count the number of houses for each ocean proximity category
run_query("""SELECT
    ocean_proximity,
    SUM(households) AS total_houses
FROM housing
GROUP BY ocean_proximity
ORDER BY total_houses DESC;
""")


🟩 Columns:
['ocean_proximity', 'total_houses']
('<1H OCEAN', 4674364.0)
('INLAND', 3105133.0)
('NEAR OCEAN', 1318018.0)
('NEAR BAY', 1106026.0)
('ISLAND', 1383.0)


In [11]:
# Calculate the median number of households for each ocean proximity category manually

run_query("""
SELECT
    ocean_proximity,
    AVG(households) AS median_households
FROM (
    SELECT
        ocean_proximity,
        households,
        ROW_NUMBER() OVER (PARTITION BY ocean_proximity ORDER BY households) as rn,
        COUNT(*) OVER (PARTITION BY ocean_proximity) as cnt
    FROM housing
) sub
WHERE rn BETWEEN (cnt + 1) / 2 AND (cnt + 2) / 2
GROUP BY ocean_proximity
ORDER BY ocean_proximity;
""")

🟩 Columns:
['ocean_proximity', 'median_households']
('<1H OCEAN', 420.0)
('INLAND', 385.0)
('ISLAND', 288.0)
('NEAR BAY', 404.5)
('NEAR OCEAN', 429.0)


In [12]:
# To check if the last function was correct or not
medians_by_region = dff.groupby('ocean_proximity')['households'].median().sort_values(ascending=False)
medians_by_region


,households
ocean_proximity,
NEAR OCEAN,429.0
<1H OCEAN,420.0
NEAR BAY,404.5
INLAND,385.0
ISLAND,288.0


In [13]:
# Calculate the average house price for each housing median age
run_query("SELECT housing_median_age, AVG(median_house_value) AS average_price FROM housing GROUP BY housing_median_age")


🟩 Columns:
['housing_median_age', 'average_price']
(1.0, 144300.0)
(2.0, 224475.91379310345)
(3.0, 235643.5806451613)
(4.0, 228652.16315789474)
(5.0, 208980.61157024794)
(6.0, 203120.40127388536)
(7.0, 193437.60693641618)
(8.0, 192058.14285714287)
(9.0, 186326.0)
(10.0, 176768.8441064639)
(11.0, 179253.99603174604)
(12.0, 181614.42372881356)
(13.0, 190476.53020134228)
(14.0, 189965.8707317073)
(15.0, 181455.44181459566)
(16.0, 200500.1496062992)
(17.0, 189967.88760806917)
(18.0, 192861.99464285714)
(19.0, 193190.41082164328)
(20.0, 195309.77056277057)
(21.0, 200450.48423423423)
(22.0, 210147.66075949368)
(23.0, 204620.05393258427)
(24.0, 205305.9619450317)
(25.0, 221508.05338078292)
(26.0, 208740.62193126022)
(27.0, 208472.8692946058)
(28.0, 208477.83333333334)
(29.0, 197414.60485651213)
(30.0, 200796.21063829787)
(31.0, 207634.13363028952)
(32.0, 202765.94285714286)
(33.0, 203077.22167487684)
(34.0, 214892.4340175953)
(35.0, 206978.77139364302)
(36.0, 208061.3773364486)
(37.0, 207630.

In [14]:
# Query to calculate the average and standard deviation of house prices.
# Since SQLite doesn't support STDDEV(), the standard deviation is calculated manually.

run_query("""SELECT
    AVG(median_house_value) AS average_price,
    -- calculate the standard deviation manually
    SQRT(
        AVG((median_house_value - (SELECT AVG(median_house_value) FROM housing)) *
            (median_house_value - (SELECT AVG(median_house_value) FROM housing)))
    ) AS std_dev_price
FROM housing """)

🟩 Columns:
['average_price', 'std_dev_price']
(206864.41315519012, 115432.8423278825)


In [15]:
# Calculate the average house price based on total number of rooms (total_rooms).
# This method groups data into bins by dividing total_rooms by 1000, then multiplying back to get bin start points.
# For each bin, the query returns:
# - The starting point of the room bin (room_bin_start)
# - The total number of households in that bin (sum of households)
# - The average house value in that bin
run_query("""
SELECT
  FLOOR(total_rooms / 1000) * 1000 AS room_bin_start,
  SUM(households) AS house_count,
  AVG(median_house_value) AS average_price
FROM housing
GROUP BY room_bin_start
ORDER BY room_bin_start;
""")



🟩 Columns:
['room_bin_start', 'house_count', 'average_price']
(0.0, 317944.0, 173451.3881118881)
(1000.0, 2199575.0, 189985.33945084648)
(2000.0, 2615815.0, 215682.07783448778)
(3000.0, 1727353.0, 229229.40681137724)
(4000.0, 983453.0, 236830.59934318555)
(5000.0, 611166.0, 233352.43349753696)
(6000.0, 460015.0, 239305.91603053434)
(7000.0, 277504.0, 238064.24651162792)
(8000.0, 170937.0, 228443.13821138212)
(9000.0, 181035.0, 230777.44347826086)
(10000.0, 122140.0, 243869.14084507042)
(11000.0, 72365.0, 256033.4871794872)
(12000.0, 64944.0, 203460.60606060605)
(13000.0, 50695.0, 193843.47826086957)
(14000.0, 54317.0, 270040.95454545453)
(15000.0, 40814.0, 269453.0588235294)
(16000.0, 49550.0, 252421.1052631579)
(17000.0, 41297.0, 245293.4)
(18000.0, 27425.0, 284200.1111111111)
(19000.0, 11250.0, 209350.0)
(20000.0, 26835.0, 180975.0)
(21000.0, 19503.0, 218900.0)
(22000.0, 3258.0, 289600.0)
(23000.0, 11641.0, 235800.0)
(24000.0, 2221.0, 239300.0)
(25000.0, 12737.0, 217000.0)
(26000.0, 

In [16]:
# Calculate the average house price based on total number of rooms.
# This method uses NTILE(10) to divide all rows into 10 equal-sized groups (deciles)
# based on the value of total_rooms, regardless of the actual room counts.
# Each group (rooms_bin) contains roughly the same number of rows.
# Then, it calculates the average price for each bin.
run_query("""
SELECT
    rooms_bin,
    AVG(median_house_value) AS average_price
FROM (
    SELECT
        median_house_value,
        NTILE(10) OVER (ORDER BY total_rooms) AS rooms_bin
    FROM housing
) sub
GROUP BY rooms_bin
ORDER BY rooms_bin;
""")



🟩 Columns:
['rooms_bin', 'average_price']
(1, 173612.46086105675)
(2, 177163.334148728)
(3, 189483.30675146772)
(4, 196642.62163485072)
(5, 203899.4610866373)
(6, 208850.30690161526)
(7, 222480.035731767)
(8, 225044.94909446893)
(9, 235444.1174743025)
(10, 236062.85952031327)


In [17]:
# Calculate the average house price based on population size in the area.
# This approach uses NTILE(10) to divide all rows into 10 equal-sized groups
# according to population values (sorted in ascending order).
# Each population_bin contains approximately the same number of rows,
# regardless of the actual population values.
# Then, it computes the average house price for each bin.
run_query("""
SELECT
    population_bin,
    AVG(median_house_value) AS average_price
FROM (
    SELECT
        median_house_value,
        NTILE(10) OVER (ORDER BY population) AS population_bin
    FROM housing
) sub
GROUP BY population_bin
ORDER BY population_bin;
""")


🟩 Columns:
['population_bin', 'average_price']
(1, 205399.09246575343)
(2, 218976.54305283757)
(3, 214782.94080234834)
(4, 212428.34654919236)
(5, 209667.93392070485)
(6, 202101.9011257954)
(7, 202726.7170827215)
(8, 199882.35535976506)
(9, 201122.1987273617)
(10, 201547.01517376408)


In [18]:
# Check how many rows (houses) are in each population bin after using NTILE(10).
# This helps verify that the data has been evenly distributed into 10 groups
# based on population, with each group containing approximately the same number of rows.
run_query("""
    SELECT
        population_bin,
        COUNT(*) AS house_count
    FROM (
        SELECT
            NTILE(10) OVER (ORDER BY population) AS population_bin
        FROM housing
    ) sub
    GROUP BY population_bin
    ORDER BY population_bin;
""")


🟩 Columns:
['population_bin', 'house_count']
(1, 2044)
(2, 2044)
(3, 2044)
(4, 2043)
(5, 2043)
(6, 2043)
(7, 2043)
(8, 2043)
(9, 2043)
(10, 2043)


In [19]:
# Alternative Approach:
# ---------------------
# Instead of executing single queries one by one, I implemented a function
# that accepts multiple SQL queries at once. It runs each query using pandas'
# read_sql_query method, which returns the results as DataFrames.
# This approach combines the power of SQL with the flexibility of Python,
# making it easier to handle and analyze multiple query results efficiently.

In [20]:
# This approach combines the readability of Python with the power of SQL.
def run_sqls(*queries):
    results = []
    for q in queries:
        df = pd.read_sql_query(q, conn)  # Run each SQL query and return result as a DataFrame
        results.append(df)               # Store the result in the results list
    return results                       # Return all query results as a list of DataFrames


In [21]:
# Example usage of the run_sqls function:
# --------------------------------------
# Define multiple SQL queries to be executed:
# - q1: Counts total number of rows in the 'housing' table.
# - q2: Calculates the average median income.
# - q3: Retrieves the first 3 rows from the 'housing' table.

q1 = "SELECT COUNT(*) FROM housing"
q2 = "SELECT AVG(median_income) FROM housing"
q3 = "SELECT * FROM housing LIMIT 3"

# Pass these queries together to run_sqls, which executes them all,
# returning a list of DataFrames — one per query.

# The results can then be iterated over or accessed individually for analysis.
results = run_sqls(q1, q2, q3)



In [22]:
# Display the results of each executed query:
for i, df in enumerate(results):
    print(f"Result {i+1}:")
    display(df)

Result 1:


,COUNT(*)
0,20433


Result 2:


,AVG(median_income)
0,3.871162


Result 3:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY


In [23]:
# Close the database connection when all queries and operations are done
conn.close()